In [1]:
# === CELL 0: DIAGNOSTIC ===
import requests
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

def test_subreddit_exists(name):
    try:
        url = f"https://www.reddit.com/r/{name}/about.json"
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if "data" in data and not data["data"].get("over18", False):  # just checking it loaded
                print(f"✓ r/{name} exists")
                return True
        elif res.status_code == 429:
            print("Rate limited by Reddit. Waiting for 60 seconds before retrying...")
            time.sleep(60)
            return test_subreddit_exists(name)  # retry after waiting
    except Exception:
        pass
    print(f"✗ r/{name} DOES NOT exist or is private")
    return False

print("Testing 3 example cities:")
test_subreddit_exists("Mumbai")
time.sleep(2)
test_subreddit_exists("Bengaluru")
time.sleep(2)
test_subreddit_exists("Jaipur")

Testing 3 example cities:
✓ r/Mumbai exists
✓ r/Bengaluru exists
✓ r/Jaipur exists


True

In [2]:
# === CELL 1: IMPORTS + CONFIGURATION ===
import requests
import pandas as pd
import json
import os
import re
import time
import random
from datetime import datetime
import datetime as dt
import xlrd

COLLECTOR_NAME = "Soubhik Sarkar"
OUTPUT_CSV = "data/reddit_cities_part2_complaints.csv"
OUTPUT_JSON = "data/reddit_cities_part2.json"
OUTPUT_EXCEL = "data/reddit_cities_part2_complaints.xlsx"
TXT_DIR = "data/txt"

os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

CITIES_PART2 = []
try:
    wb = xlrd.open_workbook(r"C:\Users\Soubhik69420\New folder\list_of_cities_and_towns_in_india-834j.xls")
    ws = wb.sheet_by_index(0)
    for row_idx in range(1, ws.nrows):
        sno = ws.cell_value(row_idx, 0)
        city = ws.cell_value(row_idx, 1)
        state = ws.cell_value(row_idx, 2)
        if city and sno:
            sno = int(sno)
            if 153 <= sno <= 305:
                CITIES_PART2.append({"city": city.strip(), "state": state.strip()})
    if CITIES_PART2:
        print(f"Loaded {len(CITIES_PART2)} cities for Part 2 (153 to 305)")
        print(f"First: {CITIES_PART2[0]['city']} | Last: {CITIES_PART2[-1]['city']}")
except Exception as e:
    print(f"Failed to load cities from excel: {e}")

next_id = 1
start_id = next_id
ID_PREFIX = "NARCF3"
print(f"Starting Unique ID from: {ID_PREFIX}-{next_id:05d}")

# ── Paths & APIs ───────────────────────────────────────────────────────────
PULLPUSH_URL = "https://api.pullpush.io/reddit/search/submission/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

START_TIMESTAMP = int(dt.datetime(2014, 1, 1).timestamp())
END_TIMESTAMP = int(dt.datetime(2025, 12, 31).timestamp())

Loaded 153 cities for Part 2 (153 to 305)
First: Nagercoil | Last: Anantnag
Starting Unique ID from: NARCF3-00001


In [3]:
# === CELL 2: KEYWORD TAXONOMY ===
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [4]:
# === CELL 3: HELPER FUNCTIONS ===
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["i lost", "i was scammed", "they took", "money deducted",
                                       "i got cheated", "duped", "fell for", "lost money",
                                       "my account", "amount debited"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["almost", "tried to scam", "i did not", "i refused",
                                         "i avoided", "beware", "warning", "how i avoided",
                                         "did not share", "suspicious"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def safe_save(df, csv_path, json_path, json_data):
    try:
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)
        print(f"  ✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"  ⚠️ Checkpoint save failed (data still in memory): {e}")

def fetch_reddit_posts(keyword, subreddit, before=None, size=100):
    """Fetch posts from PullPush API for a keyword in a subreddit."""
    params = {
        "q": keyword,
        "subreddit": subreddit,
        "size": size,
        "after": START_TIMESTAMP,
        "sort": "desc",
        "sort_type": "created_utc"
    }
    if before:
        params["before"] = before

    try:
        time.sleep(random.uniform(1.0, 3.0))
        response = requests.get(PULLPUSH_URL, params=params, headers=HEADERS, timeout=15)
        if response.status_code == 429:
            print("Rate limited. Sleeping 10 seconds...")
            time.sleep(10)
            return []
        if response.status_code != 200:
            print(f"    Failed: status {response.status_code}")
            return []
        data = response.json()
        posts = data.get("data", [])
        return posts
    except Exception as e:
        print(f"    Error fetching '{keyword}' from r/{subreddit}: {e}")
        return []

def subreddit_exists(name):
    """Return True if subreddit exists and is accessible, False otherwise."""
    try:
        url = f"https://www.reddit.com/r/{name}/about.json"
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            return True
        return False
    except Exception:
        return False

def city_to_subreddit_slug(city_name):
    """Try multiple slug formats for a city."""
    clean = city_name.strip()
    no_spaces = clean.replace(' ', '').replace('-', '').replace('(', '').replace(')', '')
    underscored = clean.replace(' ', '_').replace('-', '_')
    ALIASES = {
        "Bengaluru": ["Bengaluru", "Bangalore"],
        "Mumbai": ["Mumbai", "Bombay"],
        "Chennai": ["Chennai", "Madras"],
        "Kolkata": ["Kolkata", "Calcutta"],
        "Thiruvananthapuram": ["Thiruvananthapuram", "Trivandrum"],
        "Bhubaneswar": ["Bhubaneswar", "Bhubaneshwar"],
    }
    if clean in ALIASES:
        return ALIASES[clean] + [no_spaces, underscored]
    return [clean, no_spaces, underscored]

In [5]:
# === CELL 4: SUBREDDIT DISCOVERY ===
working_cities_p2 = {}  # {city_name: subreddit_slug}
skipped_cities = []

print("Checking for working subreddits...")
for item in CITIES_PART2:
    city = item['city']
    slugs = city_to_subreddit_slug(city)
    found = False
    for slug in slugs:
        if subreddit_exists(slug):
            working_cities_p2[city] = slug
            print(f"  ✓ r/{slug} exists ({city}, {item['state']})")
            found = True
            break
        time.sleep(0.5)
    if not found:
        skipped_cities.append(city)

print(f"\nFound {len(working_cities_p2)} working subreddits out of {len(CITIES_PART2)} cities")
print(f"Skipped {len(skipped_cities)} cities with no subreddit")

Checking for working subreddits...
  ✓ r/Thanjavur exists (Thanjavur, Tamil Nadu)
  ✓ r/MurwaraKatni exists (Murwara (Katni), Madhya Pradesh)
  ✓ r/Sambhal exists (Sambhal, Uttar Pradesh)
  ✓ r/Nadiad exists (Nadiad, Gujarat)
  ✓ r/Yamunanagar exists (Yamunanagar, Haryana)
  ✓ r/EnglishBazar exists (English Bazar, West Bengal)
  ✓ r/Eluru exists (Eluru, Andhra Pradesh)
  ✓ r/Panchkula exists (Panchkula, Haryana)
  ✓ r/Raayachuru exists (Raayachuru, Karnataka)
  ✓ r/Panvel exists (Panvel, Maharashtra)
  ✓ r/Deoghar exists (Deoghar, Jharkhand)
  ✓ r/Ongole exists (Ongole, Andhra Pradesh)
  ✓ r/Morena exists (Morena, Madhya Pradesh)
  ✓ r/Palakkad exists (Palakkad, Kerala)
  ✓ r/Purnia exists (Purnia, Bihar)
  ✓ r/Baharampur exists (Baharampur, West Bengal)
  ✓ r/Orai exists (Orai, Uttar Pradesh)
  ✓ r/Vellore exists (Vellore, Tamil Nadu)
  ✓ r/Singrauli exists (Singrauli, Madhya Pradesh)
  ✓ r/Mahesana exists (Mahesana, Gujarat)
  ✓ r/Silchar exists (Silchar, Assam)
  ✓ r/Sambalpur exist

In [ ]:
# === CELL 5: MAIN SCRAPING LOOP ===
existing_post_ids = set()
existing_reddit_urls = set()
existing_data_json = []
already_scraped_slugs = set()

if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        existing_data_json = json.load(f)
    for item in existing_data_json:
        if item.get("URL"):
            existing_reddit_urls.add(item["URL"])
        if item.get("Reddit Post ID"):
            existing_post_ids.add(item["Reddit Post ID"])
        
        # Track already scraped subreddits
        structured = item.get("StructuredData", {})
        source_platform = structured.get("Source Platform", "")
        if source_platform.startswith("Reddit r/"):
            already_scraped_slugs.add(source_platform.replace("Reddit r/", ""))

print(f"Loaded {len(existing_reddit_urls)} existing Reddit URLs to skip")
print(f"Found {len(already_scraped_slugs)} already scraped subreddits to skip")

new_records = []
new_results_count = 0
today_date = datetime.now().strftime("%Y-%m-%d")

for city_name, slug in working_cities_p2.items():
    if slug in already_scraped_slugs:
        print(f"\n=== Skipping City: {city_name} (r/{slug}) - already partially/fully scraped ===")
        continue

    print(f"\n=== Scraping City: {city_name} (r/{slug}) ===")
    
    for parent_category, subcategories in KEYWORD_TAXONOMY.items():
        for subcat, keyword_list in subcategories.items():
            for keyword in keyword_list:
                print(f"  Keyword: '{keyword}'")
                before = END_TIMESTAMP
                
                while True:
                    posts = fetch_reddit_posts(keyword, slug, before=before, size=100)
                    if not posts:
                        break
                        
                    batch_valid = 0
                    for post in posts:
                        post_id = post.get('id', '')
                        if not post_id or post_id in existing_post_ids:
                            continue
                            
                        permalink = post.get('permalink', '')
                        full_url = f"https://www.reddit.com{permalink}"
                        if full_url in existing_reddit_urls:
                            continue
                            
                        existing_post_ids.add(post_id)
                        existing_reddit_urls.add(full_url)
                            
                        created_utc = post.get('created_utc')
                        if created_utc is None:
                            continue
                        try:
                            created_utc = int(float(str(created_utc)))
                        except (ValueError, TypeError):
                            continue
                            
                        readable_date = datetime.fromtimestamp(created_utc).strftime("%Y-%m-%d")
                        selftext = post.get('selftext', '').strip()
                        title = post.get('title', '')
                        author = post.get('author', 'unknown')
                        score = post.get('score', 0)
                        
                        if not selftext or selftext in ['[deleted]', '[removed]']:
                            post_body = f"[No body text — title only]\n\nTitle contains full narrative:\n{title}"
                        else:
                            post_body = clean_text(selftext)
                            
                        assigned_id = f"{ID_PREFIX}-{next_id:05d}"
                        txt_filename = f"{assigned_id}.txt"
                        next_id += 1
                        
                        # Save TXT
                        txt_content = f"SOURCE: Reddit - r/{slug}\n"
                        txt_content += f"CITY: {city_name}\n"
                        txt_content += f"TITLE: {title}\n"
                        txt_content += f"AUTHOR: {author}\n"
                        txt_content += f"DATE: {readable_date}\n"
                        txt_content += f"URL: {full_url}\n"
                        txt_content += "--- POST TEXT ---\n\n"
                        txt_content += post_body
                        
                        full_content = title + ' ' + post_body
                        narrative_type = classify_narrative_type(full_content)
                        
                        with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f_txt:
                            f_txt.write(txt_content)
                            
                        record = {
                            "Reddit Post ID": post_id,
                            "Unique ID": assigned_id,
                            "Date of Collection": today_date,
                            "Collector Name": "Soubhik Sarkar",
                            "Source Platform": f"Reddit r/{slug}",
                            "Source Publication": "reddit.com",
                            "Original Date": readable_date,
                            "Title/Headline": title,
                            "URL": full_url,
                            "Search Query Used": keyword,
                            "Fraud Category": parent_category,
                            "Fraud Subcategory": subcat,
                            "Narrative Type": narrative_type,
                            "TXT File Name": txt_filename,
                            "Notes": f"City: {city_name} | Author: u/{author} | Score: {score} | Reddit ID: {post_id}"
                        }
                        
                        raw_item = {
                            "Reddit Post ID": post_id,
                            "URL": full_url,
                            "Title/Headline": title,
                            "Original Date": readable_date,
                            "Author": author,
                            "StructuredData": record
                        }
                        
                        new_records.append(record)
                        existing_data_json.append(raw_item)
                        new_results_count += 1
                        batch_valid += 1
                        
                    if len(posts) > 0:
                        new_before = int(posts[-1].get('created_utc', before))
                        if new_before >= before:
                            new_before = before - 1
                        before = new_before
                    else:
                        break
                        
                    print(f"    Added {batch_valid} posts, moving 'before' to {before}")
                    
                    if new_results_count % 50 == 0 and new_results_count > 0:
                        df_temp = pd.DataFrame([x["StructuredData"] for x in existing_data_json])
                        safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, existing_data_json)

print(f"\nScraping phase complete.")

Loaded 0 existing Reddit URLs to skip

=== Scraping City: Thanjavur (r/Thanjavur) ===
  Keyword: 'cyber crime'
  Keyword: 'cybercrime'
  Keyword: 'cyber fraud'
  Keyword: 'online fraud'
  Keyword: 'internet fraud'
  Keyword: 'digital fraud'
  Keyword: 'cyber scam'
  Keyword: 'online scam'
  Keyword: 'internet scam'
  Keyword: 'online cheating'
  Keyword: 'financial fraud online'
  Keyword: 'net banking fraud'
  Keyword: 'e-banking fraud'
  Keyword: 'UPI fraud'
  Keyword: 'UPI scam'
  Keyword: 'Google Pay fraud'
  Keyword: 'PhonePe fraud'
  Keyword: 'Paytm fraud'
  Keyword: 'BHIM fraud'
  Keyword: 'QR code scam'
  Keyword: 'QR code fraud'
  Keyword: 'scan and pay fraud'
  Keyword: 'payment link fraud'
  Keyword: 'collect request scam'
  Keyword: 'fake payment link'
  Keyword: 'mobile wallet fraud'
  Keyword: 'e-wallet scam'
  Keyword: 'digital wallet fraud'
  Keyword: 'OTP fraud'
  Keyword: 'OTP scam'
  Keyword: 'OTP theft'
  Keyword: 'SIM swap fraud'
  Keyword: 'SIM cloning'
  Keyword:

KeyboardInterrupt: 

In [ ]:
# === CELL 6: FINAL SAVE ===
if new_results_count > 0:
    df_new = pd.DataFrame(new_records)
    
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        df_combined.drop_duplicates(subset=["URL"], keep="last", inplace=True)
    else:
        df_combined = df_new

    safe_save(df_combined, OUTPUT_CSV, OUTPUT_JSON, existing_data_json)
    
    try:
        df_combined.to_excel(OUTPUT_EXCEL, index=False)
        print(f"  ✅ Saved Excel to {OUTPUT_EXCEL}")
    except Exception as e:
        print(f"  ⚠️ Could not save Excel: {e}")
        
    print(f"✅ Final save complete. Total records: {len(df_combined)}")
    print(f"IDs scraped this session: {ID_PREFIX}-{start_id:05d} to {ID_PREFIX}-{next_id-1:05d}")
else:
    print("No new data to save.")